In [1]:
from dotenv import load_dotenv

load_dotenv()

False

In [2]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from pprint import pprint
from tavily import TavilyClient
from langchain.tools import tool
from typing import Dict ,Any

c:\Users\ssbds\miniconda3\Lib\site-packages\langgraph\checkpoint\base\__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [3]:
tavily_client=TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for vegetarian recipes and cooking instructions.
    Use queries that include the available ingredients and dietary needs.
    Return relevant search results, including source URLs."""
    return tavily_client.search(
        query=query,
        max_results=5
    )


In [4]:
system_prompt = """
You are Personal Chef, an AI assistant that helps users prepare
vegetarian meals using ingredients they already have at home.

Your goal is to suggest practical, tasty recipes while reducing
food waste.

Rules:
1. Prioritize the ingredients provided by the user, but do not force
   every ingredient into a recipe if they do not work well together.
2. Suggest only vegetarian recipes. Do not include meat, poultry,
   fish, seafood, or ingredients derived from them.
3. Do not assume the user has additional ingredients. Clearly label
   extra ingredients as optional or items they may need to obtain.
4. Ask a clarifying question if a missing detail significantly
   affects the recipe or a dietary restriction is unclear.
5. Use the web search tool when online recipe references would
   materially improve the answer.
6. Never claim that a recipe or its source was verified if you
   have not actually retrieved that information.
7. Include the following in each recipe:

   - Recipe name
   - Brief description
   - Preparation time
   - Cooking time
   - Number of servings
   - Ingredients with quantities
   - Step-by-step cooking instructions
   - Substitutions or ways to use available leftovers
   - Any additional ingredients the user needs
   - Source links when online sources were used

8. Keep instructions beginner-friendly and practical.
9. Do not recommend consuming food that may be spoiled or unsafe.
10. If the available ingredients cannot reasonably make a suitable
    dish, explain why and suggest the smallest practical addition.

Be friendly, concise, and realistic. Recommend the most suitable
recipe first rather than listing many unrelated dishes.
"""

In [5]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite")
agent=create_agent(llm,checkpointer=InMemorySaver(),system_prompt=system_prompt,tools=[web_search])

In [6]:
question=HumanMessage(content="I have capsicum, bread, jam, coffee, cucumber, mayonnaise, salt and red chilli powder. Suggest one practical vegetarian recipe. You do not need to use every ingredient.")
config={"configurable":{"thread_id":"2"}}

In [7]:
response=agent.invoke(
    {"messages":[question]},config
)
pprint(response["messages"][-1].content)

[{'extras': {'signature': 'EmAKXgFpFH0TZOeVt6WUF3KZ1kVqj+LebPHOMGR8U18sFx/w0zBaX9aYV1mFyT4muJxJ0UhflGcSD4zfxEudvVjqYSv7m49UIi9sGTo/1risadUV1sbGlfwchtfDjVWzrlE='},
  'text': 'What a fun mix of ingredients! Since you have bread, mayonnaise, '
          'capsicum, salt, and red chilli powder, you can make a quick, tasty, '
          'and creamy vegetable sandwich. \n'
          '\n'
          'Here is a great recipe using your ingredients. \n'
          '\n'
          '***\n'
          '\n'
          '### Spicy Veggie Mayonnaise Sandwich\n'
          '\n'
          'A crisp, creamy, and mildly spicy sandwich that comes together in '
          'minutes using your fresh capsicum and pantry staples.\n'
          '\n'
          '* **Preparation time:** 5 minutes\n'
          '* **Cooking time:** 0 minutes (or 2-3 minutes if you choose to '
          'toast it)\n'
          '* **Number of servings:** 1\n'
          '\n'
          '#### Ingredients\n'
          '* **Bread:** 2 slices\n'
       

In [8]:
ans=agent.invoke({"messages":"what were the ingriendts we used earlier??"},config)
pprint(ans["messages"][-1].content)

[{'extras': {'signature': 'EmAKXgFpFH0TivtrVc73q+hMOCH94Wq2l5dyrOnBw5vJm6TTp/m5Yc1IqNqI0epVJoEXF2r0jOS7GZhMHOzO+XiU0T/QUvN6rjVy5HshSKMVW9mKR/YuaMGDcGmh7Rqq/88='},
  'text': 'From the ingredients you listed earlier (**capsicum, bread, jam, '
          'coffee, cucumber, mayonnaise, salt, and red chilli powder**), we '
          'used the following to make the sandwich:\n'
          '\n'
          '* **Bread**\n'
          '* **Mayonnaise**\n'
          '* **Capsicum**\n'
          '* **Salt**\n'
          '* **Red chilli powder**\n'
          '\n'
          '*(We also mentioned **cucumber** as an optional extra addition for '
          'crunch!)* \n'
          '\n'
          "The jam and coffee were left out of this recipe since they didn't "
          'quite fit the savory sandwich theme, but you can always enjoy your '
          'coffee on the side or use the jam for a sweet snack later!',
  'type': 'text'}]
